[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MrEGAMERZ/netguard-ai/blob/main/NetGuard_AI_Intrusion_Detection.ipynb)

# NetGuard AI: Network Intrusion Detection System

This notebook trains and compares two machine learning models (Random Forest and a Multi Layer Perceptron neural network) to classify network flow records as either Benign or Attack using the CIC-IDS2017 dataset.

To enable GPU acceleration for the neural network, go to **Runtime** -> **Change runtime type** -> select **T4 GPU**.

## 1. Install Dependencies

In [ ]:
!pip install -q datasets
print("Dependencies installed.")

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print(f"Libraries imported. Running on: {device}")

## 3. Load CIC-IDS2017 Dataset

Streaming a 200,000 record sample from Hugging Face (`muzom/CIC-IDS2017`).

In [ ]:
print("Loading dataset...")

dataset = load_dataset(
    "muzom/CIC-IDS2017",
    split="train",
    streaming=True,
)

sample = dataset.shuffle(
    seed=RANDOM_STATE,
    buffer_size=50_000,
).take(200_000)

df = pd.DataFrame(list(sample))
df.columns = df.columns.str.strip()

print(f"Dataset loaded successfully: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head(3)

## 4. Exploratory Data Analysis

In [ ]:
print(f"Total features: {len(df.columns)}")
print(df.columns.tolist()[:20], "...")

In [ ]:
df.info()

In [ ]:
print("Missing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False).head(10))
print()
print("Original class distribution:")
print(df["Label"].value_counts())

In [ ]:
label_counts = df["Label"].value_counts()

plt.figure(figsize=(12, 5))
plt.bar(label_counts.index, label_counts.values, color="#4a90d9", edgecolor="black", linewidth=0.4)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.ylabel("Count")
plt.title("Original Traffic Distribution (Sample of 200,000 Records)")
plt.tight_layout()
plt.show()

## 5. Data Cleaning and Preprocessing

Converting attack categories into binary labels (Benign = 0, Attack = 1), replacing infinite values, and selecting numeric flow metrics.

In [ ]:
# Binary mapping: Benign = 0, Attack = 1
df["binary_label"] = df["Label"].apply(
    lambda x: 0 if str(x).strip().upper() == "BENIGN" else 1
)

benign_cnt = (df["binary_label"] == 0).sum()
attack_cnt = (df["binary_label"] == 1).sum()
total = len(df)

print(f"Benign: {benign_cnt:,} ({benign_cnt/total*100:.1f}%)")
print(f"Attack: {attack_cnt:,} ({attack_cnt/total*100:.1f}%)")

plt.figure(figsize=(5, 4))
plt.bar(["Benign", "Attack"], [benign_cnt, attack_cnt], color=["#2ca02c", "#d62728"], edgecolor="black", linewidth=0.5)
plt.ylabel("Count")
plt.title("Binary Target Distribution")
for i, v in enumerate([benign_cnt, attack_cnt]):
    plt.text(i, v + 2000, f"{v:,}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if "binary_label" in numeric_cols:
    numeric_cols.remove("binary_label")

# Replace inf with nan, then fill with column medians
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

valid_cols = [c for c in numeric_cols if df[c].isnull().sum() == 0]
print(f"Valid numeric features: {len(valid_cols)}")

In [ ]:
preferred_features = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Mean",
    "Packet Length Mean",
    "Packet Length Std",
    "Average Packet Size",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "ACK Flag Count",
    "SYN Flag Count",
    "PSH Flag Count",
    "RST Flag Count",
    "Active Mean",
    "Idle Mean",
]

selected_features = [f for f in preferred_features if f in valid_cols]
if len(selected_features) < 10:
    selected_features = valid_cols

print(f"Selected {len(selected_features)} features for training:")
print(selected_features)

## 6. Train Test Split and Feature Scaling

80% training set and 20% test set using stratified sampling, followed by Standard Scaling.

In [ ]:
X = df[selected_features].values
y = df["binary_label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Testing set:  {X_test.shape[0]:,} samples")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 7. Random Forest Classifier

In [ ]:
print("Training Random Forest model...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rf_model.fit(X_train_scaled, y_train)
print("Done.")

In [ ]:
rf_predictions = rf_model.predict(X_test_scaled)

rf_accuracy  = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions, zero_division=0)
rf_recall    = recall_score(y_test, rf_predictions, zero_division=0)
rf_f1        = f1_score(y_test, rf_predictions, zero_division=0)

print(f"Accuracy:  {rf_accuracy * 100:.2f}%")
print(f"Precision: {rf_precision * 100:.2f}%")
print(f"Recall:    {rf_recall * 100:.2f}%")
print(f"F1 Score:  {rf_f1 * 100:.2f}%")
print()
print("Classification Report:")
print(classification_report(y_test, rf_predictions, target_names=["Benign", "Attack"]))

In [ ]:
cm_rf = confusion_matrix(y_test, rf_predictions)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Benign", "Attack"], yticklabels=["Benign", "Attack"])
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
importances = rf_model.feature_importances_
top_n = min(15, len(selected_features))
indices = np.argsort(importances)[::-1][:top_n]

plt.figure(figsize=(10, 5))
plt.bar(range(top_n), importances[indices], color="#4a90d9", edgecolor="black", linewidth=0.4)
plt.xticks(range(top_n), [selected_features[i] for i in indices], rotation=45, ha="right", fontsize=9)
plt.ylabel("Importance")
plt.title(f"Top {top_n} Features (Random Forest)")
plt.tight_layout()
plt.show()

## 8. Multi Layer Perceptron (Neural Network)

In [ ]:
n_features = X_train_scaled.shape[1]

nn_model = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.30),
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.20),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
], name="NetGuard_MLP")

nn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

nn_model.summary()

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1,
)

history = nn_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=25,
    batch_size=512,
    callbacks=[early_stopping, reduce_lr],
    verbose=1,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["accuracy"], label="Train Acc")
ax1.plot(history.history["val_accuracy"], label="Val Acc", linestyle="--")
ax1.set_title("Neural Network Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history.history["loss"], label="Train Loss")
ax2.plot(history.history["val_loss"], label="Val Loss", linestyle="--")
ax2.set_title("Neural Network Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Binary Cross-Entropy")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
nn_probs = nn_model.predict(X_test_scaled, verbose=0).flatten()
nn_predictions = (nn_probs >= 0.5).astype(int)

nn_accuracy  = accuracy_score(y_test, nn_predictions)
nn_precision = precision_score(y_test, nn_predictions, zero_division=0)
nn_recall    = recall_score(y_test, nn_predictions, zero_division=0)
nn_f1        = f1_score(y_test, nn_predictions, zero_division=0)

print(f"Accuracy:  {nn_accuracy * 100:.2f}%")
print(f"Precision: {nn_precision * 100:.2f}%")
print(f"Recall:    {nn_recall * 100:.2f}%")
print(f"F1 Score:  {nn_f1 * 100:.2f}%")
print()
print("Classification Report:")
print(classification_report(y_test, nn_predictions, target_names=["Benign", "Attack"]))

cm_nn = confusion_matrix(y_test, nn_predictions)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_nn, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["Benign", "Attack"], yticklabels=["Benign", "Attack"])
plt.title("Neural Network Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 9. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "Neural Network (MLP)"],
    "Accuracy (%)":  [round(rf_accuracy  * 100, 2), round(nn_accuracy  * 100, 2)],
    "Precision (%)": [round(rf_precision * 100, 2), round(nn_precision * 100, 2)],
    "Recall (%)":    [round(rf_recall    * 100, 2), round(nn_recall    * 100, 2)],
    "F1 Score (%)":  [round(rf_f1        * 100, 2), round(nn_f1        * 100, 2)],
}).set_index("Model")

print(comparison.to_string())

In [ ]:
metrics = ["Accuracy (%)", "Precision (%)", "Recall (%)", "F1 Score (%)"]
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, comparison.loc["Random Forest", metrics], width, label="Random Forest", color="#4a90d9", edgecolor="black", linewidth=0.4)
bars2 = ax.bar(x + width/2, comparison.loc["Neural Network (MLP)", metrics], width, label="Neural Network (MLP)", color="#e67e22", edgecolor="black", linewidth=0.4)

ax.set_ylabel("Score (%)")
ax.set_title("Random Forest vs Neural Network Comparison")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(80, 101)
ax.legend()
ax.bar_label(bars1, fmt="%.1f%%", padding=3, fontsize=9)
ax.bar_label(bars2, fmt="%.1f%%", padding=3, fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Sample Prediction Demo

In [ ]:
def predict_traffic(record_index=None):
    if record_index is None:
        record_index = np.random.randint(0, len(X_test))

    record = X_test_scaled[record_index].reshape(1, -1)
    actual = "Attack" if y_test[record_index] == 1 else "Benign"

    rf_pred = rf_model.predict(record)[0]
    rf_conf = rf_model.predict_proba(record)[0][rf_pred]
    nn_prob = float(nn_model.predict(record, verbose=0)[0][0])
    nn_pred = int(nn_prob >= 0.5)

    rf_label = "Attack" if rf_pred == 1 else "Benign"
    nn_label = "Attack" if nn_pred == 1 else "Benign"

    print(f"Sample Index: {record_index}")
    print(f"Actual:       {actual}")
    print(f"Random Forest: {rf_label} ({rf_conf*100:.1f}% confidence)")
    print(f"Neural Net:    {nn_label} ({nn_prob*100:.1f}% attack probability)")
    print()

In [ ]:
# Run prediction on 5 random samples
for _ in range(5):
    predict_traffic()